## JSON Generation

These modules of code is using to create the jsonl containing Q/A pairs. So far 694 pairs have been generated using NotebookLM.
Since this is the first iteration of, the number stays the same. 
Further numbers can be improved by adding more books or prompting to NotebookLM.

In [5]:
import json
import glob

all_data = []

for file in sorted(glob.glob("dataset/*.json")):

    with open(file, "r", encoding="utf-8") as f:

        data = json.load(f)

        all_data.extend(data)

print("Total QA pairs:", len(all_data))

Total QA pairs: 694


In [6]:
seen = set()
filtered = []

for item in all_data:

    q = item["instruction"].strip().lower()

    if q not in seen:
        filtered.append(item)
        seen.add(q)

print("Remaining:", len(filtered))

Remaining: 693


In [7]:
cleaned = []

for item in filtered:

    q = item["instruction"].strip()
    a = item["response"].strip()

    if len(q) < 15:
        continue

    if len(a) < 40:
        continue

    if "I don't know" in a:
        continue

    if "not enough information" in a.lower():
        continue

    cleaned.append(item)

print(len(cleaned))

690


In [8]:
import random

for sample in random.sample(cleaned,5):

    print("="*80)

    print(sample["instruction"])

    print()

    print(sample["response"])

What is the mathematical interpretation of the Finite Element Method (FEM)?

Mathematically, FEM is a procedure for obtaining numerical approximations to the solution of boundary value problems. The domain is replaced by disjoint subdomains (elements), and unknown functions are locally approximated using interpolation formulas (shape functions) governed by unit node values. The node values are then determined by minimizing a functional, such as through the Ritz or Galerkin methods.
Why do ships with open cross-sections, like containerships, experience severe torsional warping?

Open thin-walled sections possess significantly less torsional stiffness than closed sections because they cannot develop a continuous shear flow around the contour. Consequently, twisting moments cause the open cross-section to warp, meaning initially plane sections distort longitudinally out of their plane, inducing severe warping stresses.
What is shear lag in ship structural mechanics?

Shear lag is a phenom

In [9]:
import json

with open("naval_qa.jsonl","w",encoding="utf-8") as f:

    for item in cleaned:

        row = {

            "messages":[

                {
                    "role":"user",
                    "content":item["instruction"]
                },

                {
                    "role":"assistant",
                    "content":item["response"]
                }

            ]
        }

        f.write(json.dumps(row,ensure_ascii=False)+"\n")

In [10]:
count = 0

with open("naval_qa.jsonl","r",encoding="utf-8") as f:

    for line in f:
        count += 1

print(count)

690


## Training?

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
%cd /content/drive/MyDrive
!ls

/content/drive/MyDrive
 20220414_162844.jpg
'25 interns.xlsx'
 ACADEMICS
'AI Guild Code Report.gdoc'
'APPLICATION FOR NA DEPUTY COORDINATOR 2024-25.gdoc'
 ARJUNAAR_CONSULT_FIXED_RESUME.pdf
'ARJUNAAR_CORE_FIXED_RESUME (2).pdf'
 arjunaar@gmail.com_resume.gdoc
 ArjunAAR_IITMadras_9003866112_Data_Scientist.mp4
'ArjunAAR_IITMadras_9003866112_Software_Engineer (1).mp4'
 ArjunAAR_IITMadras_9003866112_Software_Engineer.mp4
 ARJUNAAR_IITM_IDCARD.pdf
 ARJUNAAR_MASTER_RESUME_NA23B006.docx
 ARJUNAAR_MASTER_RESUME_NA23B006.pdf
'ARJUNAAR_NA23B006 (1) (1).pdf'
 ArjunAAR-na23b006-1740364394823-CV.pdf
'ARJUNAAR_NA23B006 (1).pdf'
'ARJUNAAR_NA23B006 (2).pdf'
'ARJUNAAR_NA23B006 (3).pdf'
'ARJUNAAR_NA23B006 (4).pdf'
'ARJUNAAR_NA23B006 (5).pdf'
'ARJUNAAR_NA23B006 (6).pdf'
'ARJUNAAR_NA23B006 (7).pdf'
 ARJUNAAR_NA23B006_ASSIGNMENT1.pdf
 ARJUNAAR_NA23B006_LR1.py
 ARJUNAAR_NA23B006_LR2.py
 ARJUNAAR_NA23B006.pdf
 ARJUNAAR_PASSPORT_PHOTO.JPEG
 ARJUNAAR_RESUME_NA23B006_2.pdf
'ARJUNAAR_RESUME_NA23B006_FINAL (1) (1).

In [4]:
!pip install -q -U transformers
!pip install -q -U datasets
!pip install -q -U accelerate
!pip install -q -U peft
!pip install -q -U trl
!pip install -q -U bitsandbytes
!pip install -q -U sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 126.8 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 43.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 29.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 29.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 825.1/825.1 kB 52.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.2 MB/s eta 0:00:00:00:0100:01


In [46]:
import torch

from datasets import load_dataset

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    default_data_collator
)

from peft import (
    LoraConfig,
    get_peft_model,
    TaskType
)

from torch.utils.data import DataLoader

In [7]:
dataset=load_dataset("json",data_files="naval_qa.jsonl",split="train")
dataset

Generating train split: 0 examples [00:00, ? examples/s]

Dataset({
    features: ['messages'],
    num_rows: 690
})

In [49]:
model_id = "microsoft/Phi-3-mini-4k-instruct"

tokenizer = AutoTokenizer.from_pretrained(
    model_id,
    trust_remote_code=True
)

tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    dtype=torch.float16,
    device_map={"":0},
)

Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

In [50]:
model.gradient_checkpointing_enable()

model.config.use_cache = False

In [51]:
lora_config = LoraConfig(

    r=16,

    lora_alpha=32,

    lora_dropout=0.05,

    bias="none",

    task_type=TaskType.CAUSAL_LM,

    target_modules=[
        "qkv_proj",
        "o_proj",
        "gate_up_proj",
        "down_proj"
    ]
)

In [55]:
model = get_peft_model(
    model,
    lora_config
)

model.print_trainable_parameters()

trainable params: 25,165,824 || all params: 3,846,245,376 || trainable%: 0.6543


In [56]:
print(dataset[0])

{'messages': [{'role': 'user', 'content': 'Define primary, secondary, and tertiary structural responses in ship structures.'}, {'role': 'assistant', 'content': 'Primary response is the response of the entire hull when bending and twisting as a beam under external longitudinal distributions of vertical, lateral, and twisting loads. Secondary response comprises the stress and deflection of a single panel of stiffened plating, such as the bottom structure between two adjacent transverse bulkheads, loaded normal to its plane. Tertiary response describes the out-of-plane deflection and associated stress of an individual panel of plating bounded by stiffeners.'}]}


In [57]:
def format_chat(example):

    text = tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False
    )

    return {"text": text}

In [58]:
dataset=dataset.map(format_chat)

Map:   0%|          | 0/690 [00:00<?, ? examples/s]

In [60]:
MAX_LEN = 1024

def tokenize(example):

    tokens = tokenizer(

        example["text"],

        truncation=True,

        max_length=MAX_LEN,

        padding="max_length"
    )

    return tokens

In [61]:
tokenized_dataset=dataset.map(tokenize,remove_columns=dataset.column_names)

Map:   0%|          | 0/690 [00:00<?, ? examples/s]

In [62]:
tokenized_dataset.set_format(
    type="torch",
    columns=[
        "input_ids",
        "attention_mask"
    ]
)

In [64]:
print(tokenized_dataset[0])

{'input_ids': tensor([32010, 22402,  7601,  ..., 32000, 32000, 32000]), 'attention_mask': tensor([1, 1, 1,  ..., 0, 0, 0])}


In [65]:
from torch.utils.data import DataLoader
from transformers import default_data_collator

dataloader = DataLoader(
    tokenized_dataset,
    batch_size=2,
    shuffle=True,
    collate_fn=default_data_collator
)

In [66]:
batch = next(iter(dataloader))

print(batch.keys())
print(batch["input_ids"].shape)
print(batch["attention_mask"].shape)

dict_keys(['input_ids', 'attention_mask'])
torch.Size([2, 1024])
torch.Size([2, 1024])


In [68]:
from transformers import get_cosine_schedule_with_warmup

epochs = 3

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=2e-4,
    weight_decay=0.01
)

In [69]:
num_training_steps = epochs * len(dataloader)

scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(0.03 * num_training_steps),
    num_training_steps=num_training_steps
)

In [70]:
model.train()

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Phi3ForCausalLM(
      (model): Phi3Model(
        (embed_tokens): Embedding(32064, 3072, padding_idx=32000)
        (layers): ModuleList(
          (0-31): 32 x Phi3DecoderLayer(
            (self_attn): Phi3Attention(
              (o_proj): lora.Linear(
                (base_layer): Linear(in_features=3072, out_features=3072, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=3072, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=3072, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (qkv_pr

In [71]:
gradient_accumulation_steps = 4

In [72]:
from tqdm.auto import tqdm

epochs = 3

for epoch in range(epochs):

    model.train()

    total_loss = 0

    progress_bar = tqdm(dataloader)

    optimizer.zero_grad()

    for step, batch in enumerate(progress_bar):

        batch = {
            k: v.to(model.device)
            for k, v in batch.items()
        }

        labels = batch["input_ids"].clone()

        labels[labels == tokenizer.pad_token_id] = -100

        outputs = model(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"],
            labels=labels
        )

        loss = outputs.loss

        # Gradient Accumulation
        loss = loss / gradient_accumulation_steps

        loss.backward()

        if (step + 1) % gradient_accumulation_steps == 0:

            optimizer.step()

            scheduler.step()

            optimizer.zero_grad()

        total_loss += loss.item() * gradient_accumulation_steps

        progress_bar.set_description(
            f"Epoch {epoch+1}"
        )

        progress_bar.set_postfix(
            loss=loss.item() * gradient_accumulation_steps
        )

    avg_loss = total_loss / len(dataloader)

    print(f"\nEpoch {epoch+1} Average Loss: {avg_loss:.4f}\n")

  0%|          | 0/345 [00:00<?, ?it/s]


Epoch 1 Average Loss: 1.8640



  0%|          | 0/345 [00:00<?, ?it/s]


Epoch 2 Average Loss: 1.3687



  0%|          | 0/345 [00:00<?, ?it/s]


Epoch 3 Average Loss: 0.8898



In [73]:
model.save_pretrained("Naval_Phi3_LoRA")

tokenizer.save_pretrained("Naval_Phi3_LoRA")

('Naval_Phi3_LoRA/tokenizer_config.json',
 'Naval_Phi3_LoRA/chat_template.jinja',
 'Naval_Phi3_LoRA/tokenizer.json')

In [ ]:
from peft import PeftModel

base_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True
)

model = PeftModel.from_pretrained(
    base_model,
    "Naval_Phi3_LoRA"
)

## Utilising the trained weights

In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [6]:
%cd /content/drive/MyDrive/
!ls

/content/drive/MyDrive
 20220414_162844.jpg
'25 interns.xlsx'
 ACADEMICS
'AI Guild Code Report.gdoc'
'APPLICATION FOR NA DEPUTY COORDINATOR 2024-25.gdoc'
 ARJUNAAR_CONSULT_FIXED_RESUME.pdf
'ARJUNAAR_CORE_FIXED_RESUME (2).pdf'
 arjunaar@gmail.com_resume.gdoc
 ArjunAAR_IITMadras_9003866112_Data_Scientist.mp4
'ArjunAAR_IITMadras_9003866112_Software_Engineer (1).mp4'
 ArjunAAR_IITMadras_9003866112_Software_Engineer.mp4
 ARJUNAAR_IITM_IDCARD.pdf
 ARJUNAAR_MASTER_RESUME_NA23B006.docx
 ARJUNAAR_MASTER_RESUME_NA23B006.pdf
'ARJUNAAR_NA23B006 (1) (1).pdf'
 ArjunAAR-na23b006-1740364394823-CV.pdf
'ARJUNAAR_NA23B006 (1).pdf'
'ARJUNAAR_NA23B006 (2).pdf'
'ARJUNAAR_NA23B006 (3).pdf'
'ARJUNAAR_NA23B006 (4).pdf'
'ARJUNAAR_NA23B006 (5).pdf'
'ARJUNAAR_NA23B006 (6).pdf'
'ARJUNAAR_NA23B006 (7).pdf'
 ARJUNAAR_NA23B006_ASSIGNMENT1.pdf
 ARJUNAAR_NA23B006_LR1.py
 ARJUNAAR_NA23B006_LR2.py
 ARJUNAAR_NA23B006.pdf
 ARJUNAAR_PASSPORT_PHOTO.JPEG
 ARJUNAAR_RESUME_NA23B006_2.pdf
'ARJUNAAR_RESUME_NA23B006_FINAL (1) (1).

In [4]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

model_name="microsoft/Phi-3-mini-4k-instruct"

tokenizer=AutoTokenizer.from_pretrained(
    model_name
)

tokenizer.pad_token=tokenizer.eos_token

config.json:   0%|          | 0.00/967 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/3.44k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.94M [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/306 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/599 [00:00<?, ?B/s]

In [5]:
base_model=AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json:   0%|          | 0.00/16.5k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

In [9]:
#Loading Adapter
model=PeftModel.from_pretrained(
    base_model,
    "Naval_Phi3_LoRA"
)

model.eval()

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Phi3ForCausalLM(
      (model): Phi3Model(
        (embed_tokens): Embedding(32064, 3072, padding_idx=32000)
        (layers): ModuleList(
          (0-31): 32 x Phi3DecoderLayer(
            (self_attn): Phi3Attention(
              (o_proj): lora.Linear(
                (base_layer): Linear(in_features=3072, out_features=3072, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=3072, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=3072, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (qkv_pr

In [10]:
def ask_model(question):
    messages=[
        {
            "role":"user",
            "content":question
        }
    ]
    
    prompt=tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs=tokenizer(
        prompt,
        return_tensors="pt"
    )

    with torch.no_grad():
        outputs=model.generate(
            **inputs,
            max_new_tokens=300,
            temperature=0.3,
            do_sample=False,
            eos_token_id=tokenizer.eos_token_id
        )
    
    answer=tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    )

    return answer

In [11]:
print(ask_model("What is hull girder bending"))

[transformers] The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Hull girder bending is the continuous, normal bending moment acting on the hull girder (the primary longitudinal structure) from bow to stern, which is resisted by the hull girder's vertical web and flanges.


In [12]:
print(ask_model("Explain the assumptions of beam theory."))

Beam theory assumes that the material is homogeneous, isotropic, and behaves elastically (returns to its original shape after loading). It also assumes that plane sections remain plane (no shear lag), the material is perfectly bonded to the supports, and deflections are small.


In [15]:
print(ask_model("What are the types of structural design of a ship?"))

The structural design of a ship can be categorized into three main types: primary structure (the main hull girder), secondary structure (the supporting framework), and tertiary structure (the localized details).
